# 🐍 Day 4: The GIL

- GIL explanation
- When it matters
- Threads vs processes benchmarks
- Workarounds

## What is the GIL?

The Global Interpreter Lock (GIL) is a mutex that protects access to Python objects. Only one thread can execute Python bytecode at a time in a process.

## Why Does the GIL Exist?

Python's memory management (reference counting) is not thread-safe. The GIL simplifies the implementation and prevents race conditions on object access.

In [ ]:
import sys
print(f"GIL: One thread executes Python bytecode at a time per process")
print(f"Python version: {sys.version}")

## When Threads Work Well

I/O-bound work: network, disk, DB. Threads release the GIL during I/O operations (blocking calls), so others can run.

In [ ]:
import threading
import time

def io_bound():
    time.sleep(0.5)  # Simulates I/O - releases GIL
    return "done"

start = time.perf_counter()
threads = [threading.Thread(target=io_bound) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()
elapsed = time.perf_counter() - start
print(f"4 I/O tasks with threads: ~{elapsed:.2f}s (parallel)")
print(f"Sequential would be ~2s")

## When Threads Don't Help (CPU-bound)

CPU-bound work: pure Python computation. Only one thread runs Python bytecode at a time.

In [ ]:
import threading
import time

def cpu_bound(n):
    total = 0
    for i in range(n):
        total += i
    return total

n = 5_000_000

# Sequential
start = time.perf_counter()
cpu_bound(n)
cpu_bound(n)
seq_time = time.perf_counter() - start

# Threaded (no speedup for CPU-bound)
start = time.perf_counter()
t1 = threading.Thread(target=cpu_bound, args=(n,))
t2 = threading.Thread(target=cpu_bound, args=(n,))
t1.start()
t2.start()
t1.join()
t2.join()
thread_time = time.perf_counter() - start

print(f"Sequential: {seq_time:.3f}s")
print(f"Threaded: {thread_time:.3f}s")
print("Threads don't speed up CPU-bound work due to GIL")

## Multiprocessing Bypasses GIL

Each process has its own GIL. Use multiprocessing for CPU-bound parallelism.

In [ ]:
import multiprocessing
import time

def cpu_bound(n):
    total = 0
    for i in range(n):
        total += i
    return total

if __name__ == "__main__":
    n = 5_000_000
    start = time.perf_counter()
    with multiprocessing.Pool(2) as pool:
        pool.map(cpu_bound, [n, n])
    mp_time = time.perf_counter() - start
    print(f"Multiprocessing (2 processes): {mp_time:.3f}s")
    print("Can achieve near-linear speedup for CPU-bound tasks")

## GIL Release Points

The GIL is released during: I/O operations, time.sleep(), C extensions (NumPy, etc.) that release it explicitly.

In [ ]:
# NumPy/SciPy release GIL in many operations
# import numpy as np
# Multiple threads doing np.dot() can run in parallel

# time.sleep(0) yields to other threads (cooperative)
import threading
import time

def cooperative():
    for i in range(5):
        print(threading.current_thread().name, i)
        time.sleep(0)  # Yield

t1 = threading.Thread(target=cooperative, name="A")
t2 = threading.Thread(target=cooperative, name="B")
t1.start()
t2.start()
t1.join()
t2.join()

## Summary: When to Use What

| Workload | Use | Why |
|----------|-----|-----|
| I/O-bound | Threads | GIL released during I/O |
| CPU-bound | Multiprocessing | Separate processes, separate GILs |
| Mixed | asyncio or threads | Depends on pattern |

## Common Mistakes

- **Using threads for CPU-bound**: No speedup
- **Using multiprocessing for I/O**: Overhead; threads often better
- **Assuming GIL is removed**: Still present in CPython 3.12
- **Ignoring process spawn cost**: Multiprocessing has overhead

## Exercises

In [ ]:
# Exercise 1: Benchmark: 4 threads vs 4 processes for a CPU-bound loop. Compare times.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Benchmark: 4 threads vs sequential for 4x time.sleep(1). Show thread speedup.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Explain in a comment: why does time.sleep(0.1) in a thread allow others to run?
# YOUR CODE HERE

In [ ]:
# Exercise 4: Measure process spawn overhead: time to start and join 1 process vs 1 thread.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Create a workload that is I/O-bound (simulate with sleep). Compare 1 vs 4 threads.
# YOUR CODE HERE

In [ ]:
# Exercise 6: List 3 scenarios where threads are appropriate and 3 where multiprocessing is better.
# YOUR CODE HERE

## Key Takeaways

- GIL: one thread runs Python bytecode at a time
- I/O-bound: threads work (GIL released)
- CPU-bound: use multiprocessing
- C extensions can release GIL

## 🔜 Next: Day 5 - asyncio Basics

Tomorrow: async/await, coroutines, event loop!